In [19]:
import pandas as pd
df = pd.read_csv('trade_data/feb/turtle_trader_1_7_feb.csv')
df.head(2)
# df.info()

,ticket,opening_time_utc,closing_time_utc,type,lots,original_position_size,symbol,opening_price,closing_price,stop_loss,take_profit,commission_usd,swap_usd,profit_usd,equity_usd,margin_level,close_reason
0,1993757632,2026-02-06T19:44:00.773000,2026-02-06T19:56:30.030000,buy,0.01,0.01,XAUUSD,4952.533,4959.325,4952.811,NaN,-0.11,0,6.80,NaN,NaN,user
1,1993748170,2026-02-06T19:27:06.492000,2026-02-06T19:36:52.874000,buy,0.01,0.01,XAUUSD,4961.116,4954.084,NaN,NaN,-0.11,0,-7.04,NaN,NaN,user


## Converting the time into bd local time zone, reverse & drop unnecessary column

In [20]:
# Load & convert to datetime
df["opening_time_utc"] = pd.to_datetime(
    df["opening_time_utc"], format="mixed", utc=True)
df["closing_time_utc"] = pd.to_datetime(
    df["closing_time_utc"], format="mixed", utc=True)

# Convert UTC → Bangladesh time (Asia/Dhaka)
df["opening_time_bd"] = df["opening_time_utc"].dt.tz_convert("Asia/Dhaka")
df["closing_time_bd"] = df["closing_time_utc"].dt.tz_convert("Asia/Dhaka")

# reversed order
df_reversed = df.iloc[::-1].reset_index(drop=True)

# Drop columns
columns_to_drop = ["opening_time_utc", "closing_time_utc",
                   "swap_usd", "margin_level", "margin_level", "equity_usd","ticket"]
df_reversed.drop(columns=columns_to_drop, inplace=True)
df_reversed.head(2)

,type,lots,original_position_size,symbol,opening_price,closing_price,stop_loss,take_profit,commission_usd,profit_usd,close_reason,opening_time_bd,closing_time_bd
0,buy,0.02,0.02,BTCUSD,78940.76,78697.98,NaN,NaN,-0.32,-4.86,so,2026-02-01 10:22:49.492000+06:00,2026-02-01 11:07:36.590000+06:00
1,buy,0.02,0.02,BTCUSD,78925.25,78697.98,NaN,NaN,-0.32,-4.55,so,2026-02-01 10:51:15.551000+06:00,2026-02-01 11:07:36.591000+06:00


### Save update dataframe

In [21]:
# Save cleaned dataframe
df_reversed.to_csv("trade_data/feb/turtle_trader_1_7_feb_update.csv", index=False)

### Daily Insights

In [22]:
df_reversed["opening_time_bd"] = pd.to_datetime(df_reversed["opening_time_bd"])
df_reversed["date"] = df_reversed["opening_time_bd"].dt.date


# safety to handle nun 
cols = ["profit_usd", "commission_usd"]

df_reversed[cols] = (
    df_reversed[cols]
    .replace(["", "null", "None"], 0)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

df_reversed["net_profit"] = df_reversed["profit_usd"] + \
    df_reversed["commission_usd"]



### Mark win or loss

df_reversed["win"] = (df_reversed["net_profit"] > 0).astype(int)
df_reversed["loss"] = (df_reversed["net_profit"] < 0).astype(int)

# df_reversed.head(5)

### Generates Daily Insights

daily_stats = df_reversed.groupby("date").agg(
    total_trades=("net_profit", "count"),
    total_wins=("win", "sum"),
    total_losses=("loss", "sum"),
    total_profit=("net_profit", lambda x: x[x > 0].sum()),
    total_loss=("net_profit", lambda x: x[x < 0].sum()),
    net_pnl=("net_profit", "sum")
).reset_index()


daily_stats.to_csv("trade_data/feb/turtle_trader_1_7_feb_daily_stats.csv", index=False)

print(daily_stats)

         date  total_trades  total_wins  total_losses  total_profit  \
0  2026-02-01            26          12            14         31.89   
1  2026-02-02             7           4             3         27.79   
2  2026-02-04            28          13            15         82.18   
3  2026-02-05             1           0             1          0.00   
4  2026-02-06            20           9            11         98.09   
5  2026-02-07             3           1             2          6.69   

   total_loss  net_pnl  
0      -58.94   -27.05  
1      -20.41     7.38  
2      -95.27   -13.09  
3      -11.84   -11.84  
4      -73.27    24.82  
5       -7.44    -0.75  
